# native-indic-g2p — automated v2 dataset and T4 training\n\nThis run downloads a pinned CC0 IndicCorpV2 range, creates a strict-Devanagari word-disjoint dataset, labels it with the pinned compatibility labeler, and trains the hybrid model. It can take several hours. Select **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
!git clone --depth 1 https://github.com/manu2407/Native-indic-G2P.git /content/Native-indic-G2P\n%cd /content/Native-indic-G2P\n!pip install -q -r requirements.txt\nimport sys, torch\nassert sys.version_info < (3, 13), 'The pinned compatibility labeler requires Python 3.12 or older.'\nassert torch.cuda.is_available(), 'Enable a T4 GPU runtime before training.'\n!nvidia-smi

In [ ]:
!git clone https://github.com/somnat/Hindi-word-prosody.git vendor/Hindi-word-prosody\n!git -C vendor/Hindi-word-prosody checkout ac01da28ff2801a6a7e94829efbfe14b1c8179dd\n!python download_indiccorp_v2.py datasets/raw/indiccorp_hindi_v2_1m\n!python build_hindi_g2p_v2.py datasets/raw/indiccorp_hindi_v2_1m/hi-1.txt --upstream vendor/Hindi-word-prosody --name hindi_g2p_v2_1m

In [ ]:
!python -c "from pathlib import Path; from neural_data import verify_manifest; print(verify_manifest(Path('datasets/validation/neural_hindi_g2p_v2_1m/manifest.json'))['training'])"

In [ ]:
!python neural_g2p.py train --manifest datasets/validation/neural_hindi_g2p_v2_1m/manifest.json --device cuda --output models/neural_g2p_v2_t4.pt --epochs 1000 --patience 30 --learning-rate 2e-4 --batch-size 128 --eval-batch-size 256 --d-model 256 --heads 8 --layers 4 --feedforward 1024 --dropout 0.1 --input-mode native